# 04 · Performance & Advanced

Performance and advanced patterns: partitioning & pruning, broadcast
join tuning, diagnosing skew, caching, UDF vs native functions, and Spark SQL.

`sales` is the typed/clean fact (`read_sales_typed`).

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "utils" / "spark.py").exists() and _root != _root.parent:
    _root = _root.parent
for _p in (str(_root), str(_root / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from utils.spark import build_session, read_dim, read_sales_raw, read_sales_typed

spark = build_session("nb04")
print("Spark", spark.version, "ready")

sales = read_sales_typed(spark)
products = read_dim(spark, "dim_products").withColumn("category", F.initcap(F.trim("category")))
print("typed sales rows:", sales.count())

## Task 1: Partitioning & partition pruning

Add a `txn_month` column (`yyyy-MM`) to `sales` and write it to a temp directory **partitioned by `txn_month`** as Parquet. Provide `out_path` (the directory) and `months` (the list of `txn_month=...` partition folders). Then read it back filtering to `'2026-05'` into `one_month` — Spark prunes the other partitions.

In [ ]:
import tempfile, os
out_path = tempfile.mkdtemp(prefix="sales_part_")
(
    sales.withColumn("txn_month", F.date_format("txn_date", "yyyy-MM"))
    .write.mode("overwrite").partitionBy("txn_month").parquet(out_path)
)
months = [d for d in os.listdir(out_path) if d.startswith("txn_month=")]
one_month = spark.read.parquet(out_path).filter(F.col("txn_month") == "2026-05")
print("partitions:", len(months), "| 2026-05 rows:", one_month.count())

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert len(months) >= 6, months
assert one_month.filter(F.col("txn_month") != "2026-05").count() == 0
print("✅ Task 1 passed — partitions:", len(months))

## Task 2: Broadcast vs shuffle join

Join `sales` to `products` two ways and inspect the physical plans: `no_bc` (plain join) and `with_bc` (broadcasting `products`). Capture `with_bc`'s executed plan as the string `bc_plan` and confirm it uses a broadcast join.

In [ ]:
no_bc = sales.join(products, "product_id")
with_bc = sales.join(F.broadcast(products), "product_id")
bc_plan = with_bc._jdf.queryExecution().executedPlan().toString()
print(bc_plan[:400])

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert "BroadcastHashJoin" in bc_plan or "Broadcast" in bc_plan
print("✅ Task 2 passed — broadcast confirmed")

## Task 3: Diagnose data skew

One store dominates the sales (a classic skew problem). Compute `top_store` (the `store_id` with the most rows) and `top_share` (its fraction of all rows, 0–1). A naive `groupBy("store_id")` puts a disproportionate load on the task handling this key.

In [ ]:
_counts = sales.groupBy("store_id").count()
_total = sales.count()
_top = _counts.orderBy(F.desc("count")).first()
top_store = _top["store_id"]
top_share = _top["count"] / _total
print("skewed store:", top_store, "share:", round(top_share, 3))

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert top_store == "ST001", top_store
assert top_share > 0.1, top_share
print("✅ Task 3 passed — skew on", top_store, round(top_share, 3))

## Task 4: Cache an intermediate

Cache a filtered intermediate so repeated actions reuse it. Create `cached = sales.filter(revenue > 5)`, mark it cached, and materialize it with a `count()`. Confirm it's held in memory.

In [ ]:
cached = sales.filter(F.col("revenue") > 5).cache()
_ = cached.count()  # materialize the cache
print("in memory:", cached.storageLevel.useMemory, "| rows:", cached.count())

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert cached.storageLevel.useMemory, "DataFrame is not cached in memory"
print("✅ Task 4 passed")

## Task 5: UDF vs native expression

Label each sale's discount band: `'none'` if discount == 0, `'low'` if < 0.2, else `'high'`. Implement it **two ways** on `sales` → `banded`: a Python UDF column `band_udf` and a native `when` column `band_native`. They must agree. (Native is preferred — no Python serialization per row.)

In [ ]:
from pyspark.sql.types import StringType

def _band(d):
    if d is None:
        return None
    if d == 0:
        return "none"
    return "low" if d < 0.2 else "high"

band_udf = F.udf(_band, StringType())
banded = (
    sales
    .withColumn("band_udf", band_udf(F.col("discount")))
    .withColumn(
        "band_native",
        F.when(F.col("discount") == 0, "none")
        .when(F.col("discount") < 0.2, "low")
        .otherwise("high"),
    )
)
banded.select("discount", "band_udf", "band_native").show(5)

In [ ]:
# ✅ CHECK — run this cell to grade your answer
_mismatch = banded.filter(
    F.col("discount").isNotNull() & (F.col("band_udf") != F.col("band_native"))
).count()
assert _mismatch == 0, ("udf/native disagree on", _mismatch, "rows")
print("✅ Task 5 passed — UDF and native agree; prefer native")

## Task 6: Equivalent Spark SQL

Register `sales` as a temp view `sales_v` and write a SQL query returning per-store `total_revenue` (rounded) and `n` (txn count), top 5 by revenue. Name the result `sql_result`.

In [ ]:
sales.createOrReplaceTempView("sales_v")
sql_result = spark.sql(
    "SELECT store_id, round(sum(revenue), 2) AS total_revenue, count(*) AS n "
    "FROM sales_v GROUP BY store_id ORDER BY total_revenue DESC LIMIT 5"
)
sql_result.show()

In [ ]:
# ✅ CHECK — run this cell to grade your answer
assert sql_result.count() == 5
assert sql_result.first()["store_id"] == "ST001"
print("✅ Task 6 passed")

---
🎉 **Solution notebook** — all cells should run top to bottom.